<a href="https://colab.research.google.com/github/maryamfaghihi/data-base/blob/main/data_base_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas requests pubchempy dilimap openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 103.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 31.7 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [dilimap]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.


In [ ]:
# =============================================================
# Build a DILI modeling dataset from public structured sources
# =============================================================

# Install once:
# pip install pandas requests pubchempy dilimap openpyxl

import pandas as pd
import requests
import dilimap as dmap

# ------------------------------------------------------------
# 1. Load DILImap compound-level datasets
# ------------------------------------------------------------

dili_labels = dmap.datasets.compound_DILI_labels()
cmax = dmap.datasets.compound_Cmax_values().reset_index()
viability = dmap.datasets.compound_cell_viability().reset_index()

print("DILI labels:", dili_labels.shape)
print("Cmax:", cmax.shape)
print("Viability:", viability.shape)

print("\nDILI label columns:")
print(dili_labels.columns.tolist())

print("\nCmax columns:")
print(cmax.columns.tolist())

print("\nViability columns:")
print(viability.columns.tolist())

# ------------------------------------------------------------
# 2. Standardize column names
# ------------------------------------------------------------

def clean_cols(df):
    df = df.copy()
    df.columns = [
        str(c).strip().lower().replace(" ", "_").replace("-", "_")
        for c in df.columns
    ]
    return df

dili_labels = clean_cols(dili_labels)
cmax = clean_cols(cmax)
viability = clean_cols(viability)

# ------------------------------------------------------------
# 3. Identify a shared compound identifier
# ------------------------------------------------------------

possible_ids = [
    "compound",
    "compound_name",
    "library_cmpds",
    "name",
    "drug",
    "drug_name"
]

def find_id_col(df):
    for c in possible_ids:
        if c in df.columns:
            return c
    return None

id_dili = find_id_col(dili_labels)
id_cmax = find_id_col(cmax)
id_viab = find_id_col(viability)

print("Identifiers:", id_dili, id_cmax, id_viab)

if None in [id_dili, id_cmax, id_viab]:
    raise ValueError(
        "A common compound-name column was not found automatically. "
        "Inspect the printed columns and set the identifiers manually."
    )

dili_labels = dili_labels.rename(columns={id_dili: "compound_name"})
cmax = cmax.rename(columns={id_cmax: "compound_name"})
viability = viability.rename(columns={id_viab: "compound_name"})

# ------------------------------------------------------------
# 4. Merge DILImap compound-level information
# ------------------------------------------------------------

master = dili_labels.merge(
    cmax,
    on="compound_name",
    how="left",
    suffixes=("", "_cmax")
)

master = master.merge(
    viability,
    on="compound_name",
    how="left",
    suffixes=("", "_viability")
)

print("Merged DILImap dataset:", master.shape)

# ------------------------------------------------------------
# 5. Add PubChem identifiers and canonical SMILES
# ------------------------------------------------------------

def pubchem_properties(name):
    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{name}/property/"
        "CanonicalSMILES,IsomericSMILES,InChIKey,MolecularFormula/JSON"
    )

    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        props = r.json()["PropertyTable"]["Properties"][0]

        return pd.Series({
            "pubchem_cid": props.get("CID"),
            "canonical_smiles": props.get("ConnectivitySMILES")
                                or props.get("CanonicalSMILES"),
            "isomeric_smiles": props.get("SMILES")
                               or props.get("IsomericSMILES"),
            "inchikey": props.get("InChIKey"),
            "molecular_formula": props.get("MolecularFormula")
        })

    except Exception:
        return pd.Series({
            "pubchem_cid": None,
            "canonical_smiles": None,
            "isomeric_smiles": None,
            "inchikey": None,
            "molecular_formula": None
        })

pubchem_df = master["compound_name"].apply(pubchem_properties)

master = pd.concat(
    [master.reset_index(drop=True),
     pubchem_df.reset_index(drop=True)],
    axis=1
)

# ------------------------------------------------------------
# 6. Download Tox21 Nrf2/ARE assay data from PubChem
# AID 743203
# ------------------------------------------------------------

AID = 743203

assay_url = (
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
    f"assay/aid/{AID}/CSV"
)

response = requests.get(assay_url, timeout=60)
response.raise_for_status()

with open("tox21_nrf2_are_743203.csv", "wb") as f:
    f.write(response.content)

tox21 = pd.read_csv("tox21_nrf2_are_743203.csv", low_memory=False)

print("Tox21 assay dimensions:", tox21.shape)
print("Tox21 columns:")
print(tox21.columns.tolist())

# ------------------------------------------------------------
# 7. Keep useful assay columns
# ------------------------------------------------------------

candidate_cols = [
    "PUBCHEM_CID",
    "PUBCHEM_ACTIVITY_OUTCOME",
    "PUBCHEM_ACTIVITY_SCORE",
    "PUBCHEM_ACTIVITY_URL"
]

available_cols = [c for c in candidate_cols if c in tox21.columns]

tox21_small = tox21[available_cols].copy()

if "PUBCHEM_CID" in tox21_small.columns:
    tox21_small = tox21_small.rename(
        columns={
            "PUBCHEM_CID": "pubchem_cid",
            "PUBCHEM_ACTIVITY_OUTCOME": "nrf2_are_activity",
            "PUBCHEM_ACTIVITY_SCORE": "nrf2_are_score"
        }
    )

# ------------------------------------------------------------
# 8. Merge Tox21 with DILI dataset using PubChem CID
# ------------------------------------------------------------

master["pubchem_cid"] = pd.to_numeric(
    master["pubchem_cid"],
    errors="coerce"
)

if "pubchem_cid" in tox21_small.columns:
    tox21_small["pubchem_cid" ] = pd.to_numeric(
        tox21_small["pubchem_cid"],
        errors="coerce"
    )

    final = master.merge(
        tox21_small,
        on="pubchem_cid",
        how="left"
    )
else:
    final = master.copy()
    print(
        "PUBCHEM_CID was not found in the exported assay file. "
        "Inspect the assay columns and modify the merge step."
    )

# ------------------------------------------------------------
# 9. Distinguish tested-inactive from not tested
# ------------------------------------------------------------

if "nrf2_are_activity" in final.columns:
    final["nrf2_are_test_status"] = final[
        "nrf2_are_activity"
    ].apply(
        lambda x: "not_tested"
        if pd.isna(x)
        else str(x).lower()
    )

# ------------------------------------------------------------
# 10. Remove duplicates conservatively
# ------------------------------------------------------------

final = final.drop_duplicates(
    subset=["compound_name", "pubchem_cid"]
)

# ------------------------------------------------------------
# 11. Create a model-ready subset
# ------------------------------------------------------------

model_cols = [
    c for c in [
        "compound_name",
        "pubchem_cid",
        "inchikey",
        "canonical_smiles",
        "isomeric_smiles",
        "molecular_formula",
        "nrf2_are_activity",
        "nrf2_are_score",
        "nrf2_are_test_status"
    ]
    if c in final.columns
]

dili_related = [
    c for c in final.columns
    if "dili" in c.lower()
]

mechanistic_related = [
    c for c in final.columns
    if (
        "cmax" in c.lower()
        or "viab" in c.lower()
        or "ic10" in c.lower()
    )
]

selected_cols = list(dict.fromkeys(
    model_cols +
    dili_related +
    mechanistic_related
))

model_df = final[selected_cols].copy()

# ------------------------------------------------------------
# 12. Generate an audit summary
# ------------------------------------------------------------

audit = pd.DataFrame({
    "metric": [
        "Total DILI compounds",
        "Matched to PubChem CID",
        "Matched to Tox21 Nrf2/ARE",
        "With canonical SMILES"
    ],
    "n": [
        len(final),
        final["pubchem_cid"].notna().sum(),
        final.get(
            "nrf2_are_activity",
            pd.Series(index=final.index, dtype=object)
        ).notna().sum(),
        final["canonical_smiles"].notna().sum()
    ]
})

print("\nAudit summary:")
print(audit)

# ------------------------------------------------------------
# 13. Export
# ------------------------------------------------------------

with pd.ExcelWriter(
    "DILI_Tox21_model_dataset.xlsx",
    engine="openpyxl"
) as writer:

    final.to_excel(
        writer,
        sheet_name="Full_Merged_Data",
        index=False
    )

    model_df.to_excel(
        writer,
        sheet_name="Model_Data",
        index=False
    )

    audit.to_excel(
        writer,
        sheet_name="Audit",
        index=False
    )

print("\nSaved: DILI_Tox21_model_dataset.xlsx")

Package: s3://dilimap/public/data. Top hash: 155a2b3b63
Package: s3://dilimap/public/data. Top hash: 155a2b3b63
Package: s3://dilimap/public/data. Top hash: 155a2b3b63
DILI labels: (2472, 60)
Cmax: (345, 26)
Viability: (304, 7)

DILI label columns:
['LTKBID', 'compound_name', 'DILIrank', 'label_section', 'severity_class', 'DILIst', 'roa', 'source', 'livertox_score', 'livertox_primary_classification', 'livertox_secondary_classification', 'molecule_chembl_id', 'atc_classifications', 'availability_type', 'black_box_warning', 'chebi_par_id', 'chemical_probe', 'chirality', 'dosed_ingredient', 'first_approval', 'first_in_class', 'helm_notation', 'indication_class', 'inorganic_flag', 'max_phase', 'molecule_type', 'natural_product', 'oral', 'orphan', 'parenteral', 'polymer_flag', 'pref_name', 'prodrug', 'structure_type', 'therapeutic_flag', 'topical', 'usan_stem', 'usan_stem_definition', 'usan_substem', 'usan_year', 'withdrawn_flag', 'smiles', 'mw', 'alogp', 'warning_class', 'warning_country',